# Peer University Benchmarking Analysis

**Objective:** Compare AUB citation patterns against peer institutions (Lehigh, Marquette, Villanova)

## Data Sources:
- AUB: `data/processed/cleaned_data.csv` (already cleaned)
- Lehigh: `data/raw/lehigh.csv`
- Marquette: `data/raw/marquette.csv`
- Villanova: `data/raw/villanova.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

%matplotlib inline

## 1. Load All Data Sources

In [ ]:
# Load AUB data (already cleaned)
aub = pd.read_csv('../data/processed/cleaned_data.csv')
print(f"AUB: {len(aub)} publications")
print(f"Columns: {list(aub.columns)}")
aub.head()

In [ ]:
# Load peer universities
lehigh = pd.read_csv('../data/raw/lehigh.csv')
marquette = pd.read_csv('../data/raw/marquette.csv')
villanova = pd.read_csv('../data/raw/villanova.csv')

print(f"Lehigh: {len(lehigh)} publications")
print(f"Marquette: {len(marquette)} publications")
print(f"Villanova: {len(villanova)} publications")

## 2. Inspect Schemas

Check if column names match across all 4 sources

In [ ]:
print("Lehigh columns:")
print(list(lehigh.columns))
print("\nMarquette columns:")
print(list(marquette.columns))
print("\nVillanova columns:")
print(list(villanova.columns))

In [ ]:
# Check dtypes for each
print("=== AUB ===")
print(aub.dtypes)
print("\n=== Lehigh ===")
print(lehigh.dtypes)
print("\n=== Marquette ===")
print(marquette.dtypes)
print("\n=== Villanova ===")
print(villanova.dtypes)

## 3. Standardize Column Names

Map peer university columns to match AUB schema

In [ ]:
# TODO: Map columns based on schema inspection above
# Example (adjust based on actual column names):

# column_mapping = {
#     'Citation Count': 'citations',
#     'Publication Year': 'year',
#     'Source Title': 'venue',
#     # Add more mappings as needed
# }

# lehigh = lehigh.rename(columns=column_mapping)
# marquette = marquette.rename(columns=column_mapping)
# villanova = villanova.rename(columns=column_mapping)

print("Column standardization complete")

## 4. Add Institution Tags & Merge

In [ ]:
# Add institution column
aub['institution'] = 'AUB'
lehigh['institution'] = 'Lehigh'
marquette['institution'] = 'Marquette'
villanova['institution'] = 'Villanova'

# Select common columns (adjust based on schema)
# common_cols = ['institution', 'title', 'year', 'citations', 'venue', 'authors']

# Concatenate all
all_universities = pd.concat([
    aub,
    lehigh,
    marquette,
    villanova
], ignore_index=True)

print(f"\nTotal publications: {len(all_universities)}")
print(f"\nBreakdown by institution:")
print(all_universities['institution'].value_counts())

## 5. Save Merged Dataset

In [ ]:
# Save as parquet for fast loading
output_path = '../data/processed/all_universities.parquet'
all_universities.to_parquet(output_path, index=False)
print(f"Saved to: {output_path}")

## 6. Comparative Analysis

### 6.1 Citation Statistics

In [ ]:
# Overall citation statistics by institution
citation_stats = all_universities.groupby('institution')['citations'].agg([
    'count',
    'mean',
    'median',
    'std',
    'min',
    'max',
    ('q25', lambda x: x.quantile(0.25)),
    ('q75', lambda x: x.quantile(0.75))
]).round(2)

citation_stats

In [ ]:
# Visualization: Citation distribution comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box plot
ax1 = axes[0, 0]
all_universities.boxplot(column='citations', by='institution', ax=ax1)
ax1.set_title('Citation Distribution by Institution')
ax1.set_xlabel('Institution')
ax1.set_ylabel('Citations')
plt.sca(ax1)
plt.xticks(rotation=45)

# Violin plot
ax2 = axes[0, 1]
sns.violinplot(data=all_universities, x='institution', y='citations', ax=ax2)
ax2.set_title('Citation Density by Institution')
ax2.set_xlabel('Institution')
ax2.set_ylabel('Citations')
plt.sca(ax2)
plt.xticks(rotation=45)

# Bar chart - mean citations
ax3 = axes[1, 0]
citation_stats['mean'].plot(kind='bar', ax=ax3, color='skyblue')
ax3.set_title('Mean Citations by Institution')
ax3.set_xlabel('Institution')
ax3.set_ylabel('Mean Citations')
plt.sca(ax3)
plt.xticks(rotation=45)

# Publication counts
ax4 = axes[1, 1]
all_universities['institution'].value_counts().plot(kind='bar', ax=ax4, color='coral')
ax4.set_title('Number of Publications by Institution')
ax4.set_xlabel('Institution')
ax4.set_ylabel('Count')
plt.sca(ax4)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 6.2 Temporal Trends

In [ ]:
# Publications over time
yearly_pubs = all_universities.groupby(['institution', 'year']).size().reset_index(name='count')

plt.figure(figsize=(14, 6))
for inst in all_universities['institution'].unique():
    data = yearly_pubs[yearly_pubs['institution'] == inst]
    plt.plot(data['year'], data['count'], marker='o', label=inst)

plt.title('Publication Trends by Institution')
plt.xlabel('Year')
plt.ylabel('Number of Publications')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Average citations over time
yearly_citations = all_universities.groupby(['institution', 'year'])['citations'].mean().reset_index()

plt.figure(figsize=(14, 6))
for inst in all_universities['institution'].unique():
    data = yearly_citations[yearly_citations['institution'] == inst]
    plt.plot(data['year'], data['citations'], marker='o', label=inst)

plt.title('Average Citations per Publication by Institution Over Time')
plt.xlabel('Year')
plt.ylabel('Average Citations')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 6.3 High-Impact Publications

In [ ]:
# Define thresholds for high-impact papers
thresholds = [10, 25, 50, 100]

high_impact = pd.DataFrame()
for threshold in thresholds:
    counts = all_universities[all_universities['citations'] >= threshold].groupby('institution').size()
    high_impact[f'{threshold}+ citations'] = counts

high_impact = high_impact.fillna(0).astype(int)
high_impact

In [ ]:
# Percentage of high-impact papers
total_pubs = all_universities['institution'].value_counts()

high_impact_pct = pd.DataFrame()
for col in high_impact.columns:
    high_impact_pct[col] = (high_impact[col] / total_pubs * 100).round(2)

high_impact_pct

In [ ]:
# Visualization
high_impact_pct.plot(kind='bar', figsize=(12, 6))
plt.title('Percentage of High-Impact Publications by Institution')
plt.xlabel('Institution')
plt.ylabel('Percentage (%)')
plt.legend(title='Citation Threshold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Key Findings Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS: AUB vs Peer Universities")
print("=" * 60)
print(f"\nTotal Publications:")
print(all_universities['institution'].value_counts())
print(f"\nCitation Statistics:")
print(citation_stats[['mean', 'median', 'max']])
print(f"\nHigh-Impact Papers (100+ citations):")
print(high_impact['100+ citations'])
print(f"\nPercentage High-Impact:")
print(high_impact_pct['100+ citations'])

## Next Steps

1. Field/discipline analysis - compare citation patterns across research areas
2. Author-level metrics - h-index, productivity comparisons
3. Venue quality - journal impact factors, conference rankings
4. Collaboration patterns - co-authorship networks